# Interval prediksi (conformal) untuk regresi: "tebakan 4,2, kemungkinan besar 3,1–5,0"
**Kapan dipakai:** setelah model regresi jadi. Untuk slide (seberapa yakin model), memilih baris
yang perlu dicek manual, atau lomba yang meminta batas bawah/atas (coverage, pinball loss).
Bekerja di atas OOF regresi mana pun (`oof_<MODEL>.npy`, n × 1) tanpa melatih ulang model utama.

1. **global**: lebar = kuantil |sisa OOF|, sama untuk semua baris.
2. **ternormalisasi**: model kedua menebak "seberapa sulit" tiap baris (|sisa|). Lebar = kuantil
   (|sisa| / kesulitan) × kesulitan, jadi baris sulit dapat interval lebar.
3. **per kelompok** (Mondrian): kuantil dihitung terpisah per kelompok tebakan (`N_GROUPS`
   kuantil) atau per kolom kategori `GROUP_COL`.

Cakupan dinilai jujur: OOF dibagi dua, kalibrasi di separuh, cek di separuh lain, lalu ditukar.
Target cakupan = `1 − ALPHA`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

## Konfigurasi
`SOURCE_MODEL` = nama OOF regresi di `OOF_FOLDER`, mis. dari `stacking_teks_tabular_regresi`.
`LOG_TARGET` harus sama dengan notebook sumber (interval dihitung di skala model, lalu `expm1`).

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/interval_prediksi"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                # GANTI saat lomba
TARGET = "rating"                            # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]         # GANTI saat lomba: untuk model kesulitan
SOURCE_MODEL = "lgbm_stacking_regresi"       # GANTI: OOF regresi yang dipakai
LOG_TARGET = False                           # GANTI: samakan dengan notebook sumber
ALPHA = 0.1                                  # 0.1 → interval 90%
N_GROUPS = 5
GROUP_COL = None                             # alt: "department_name" (kelompok per kategori)
CLIP_TO_RANGE = True                         # potong interval ke rentang target train
N_FOLDS = 5
SEED = 42

## Baca data + TF-IDF (untuk model kesulitan)

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
target = train[TARGET].astype(float).to_numpy()
y = target
if LOG_TARGET:
    y = np.log1p(target)
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train))
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
vectorizer.fit(pd.concat([full_train, full_test]))
matrix_train = vectorizer.transform(full_train)
matrix_test = vectorizer.transform(full_test)
print("train:", train.shape, "| test:", test.shape, "| TF-IDF:", matrix_train.shape)

## Muat OOF sumber
Kalau file belum ada (demo), dibuat OOF Ridge cepat. Saat lomba file harus ada.

In [ ]:
def ridge_oof(labels, X, X_test):
    oof = np.zeros(X.shape[0])
    test_pred = np.zeros(X_test.shape[0])
    for idx_fit, idx_val in folds:
        fitted = Ridge(alpha=2.0).fit(X[idx_fit], labels[idx_fit])
        oof[idx_val] = fitted.predict(X[idx_val])
        test_pred += fitted.predict(X_test) / N_FOLDS
    return oof, test_pred


oof_path = f"{OOF_FOLDER}/oof_{SOURCE_MODEL}.npy"
if os.path.exists(oof_path):
    oof = np.load(oof_path).reshape(-1)
    test_pred = np.load(f"{OOF_FOLDER}/test_{SOURCE_MODEL}.npy").reshape(-1)
else:
    print("OOF sumber tidak ada → OOF Ridge demo")
    oof, test_pred = ridge_oof(y, matrix_train, matrix_test)
residual = y - oof
print("OOF:", oof.shape, "| RMSE skala model:", round(np.sqrt(np.mean(residual ** 2)), 4))

## Model kesulitan + kelompok
Kesulitan = tebakan `log(|sisa| + kecil)` dari TF-IDF + tebakan sumber (Ridge, di dalam lipatan).

In [ ]:
def with_pred(matrix, pred):
    return hstack([matrix, csr_matrix(pred[:, None])]).tocsr()


log_gap = np.log(np.abs(residual) + 0.05)
gap_oof, gap_test = ridge_oof(log_gap, with_pred(matrix_train, oof),
                              with_pred(matrix_test, test_pred))
pred_all = np.concatenate([oof, test_pred])
sigma_all = np.exp(np.concatenate([gap_oof, gap_test]))
if GROUP_COL is None:
    edges = np.quantile(oof, np.linspace(0, 1, N_GROUPS + 1)[1:-1])
    group_all = np.digitize(pred_all, edges)
else:
    groups = pd.concat([train[GROUP_COL], test[GROUP_COL]]).astype(str)
    group_all = pd.factorize(groups)[0]
n_train = len(train)
print("korelasi kesulitan vs |sisa|:", round(np.corrcoef(gap_oof, np.abs(residual))[0, 1], 3))

## Tiga cara interval
`calib` = nomor baris train untuk kalibrasi, `rows` = nomor baris yang diberi interval
(train 0..n−1, test n..). Kuantil conformal: ⌈(n+1)(1−α)⌉ / n.

In [ ]:
def conformal_quantile(scores):
    level = min(1.0, np.ceil((len(scores) + 1) * (1 - ALPHA)) / len(scores))
    return np.quantile(scores, level)


def interval_global(calib, rows):
    width = conformal_quantile(np.abs(residual[calib]))
    return pred_all[rows] - width, pred_all[rows] + width


def interval_normalized(calib, rows):
    width = conformal_quantile(np.abs(residual[calib]) / sigma_all[calib]) * sigma_all[rows]
    return pred_all[rows] - width, pred_all[rows] + width


def interval_group(calib, rows):
    width = np.zeros(len(rows))
    for group in np.unique(group_all[rows]):
        in_calib = calib[group_all[calib] == group]
        if len(in_calib) < 30:
            in_calib = calib                      # kelompok kecil → pakai semua
        width[group_all[rows] == group] = conformal_quantile(np.abs(residual[in_calib]))
    return pred_all[rows] - width, pred_all[rows] + width


METHODS = {"global": interval_global, "ternormalisasi": interval_normalized,
           "per kelompok": interval_group}
low, high = interval_global(np.arange(n_train), np.arange(n_train, len(pred_all)))
print("uji: lebar global test =", round(float(np.mean(high - low)), 4))

## Cakupan jujur (kalibrasi separuh, cek separuh lain)
"cakupan terburuk" = cakupan kelompok tebakan yang paling rendah: cara yang adaptif menjaga
angka ini tetap dekat target.

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(n_train)
first_half = order[: n_train // 2]
second_half = order[n_train // 2:]
halves = [(first_half, second_half), (second_half, first_half)]
rows = []
coverage_by_group = {}
for name in METHODS:
    inside = np.zeros(n_train, dtype=bool)
    width = np.zeros(n_train)
    for calib, check in halves:
        low, high = METHODS[name](calib, check)
        inside[check] = (y[check] >= low) & (y[check] <= high)
        width[check] = high - low
    inside_series = pd.Series(inside)
    coverage_by_group[name] = inside_series.groupby(group_all[:n_train]).mean()
    rows.append({"cakupan": round(inside.mean(), 4), "lebar rata-rata": round(width.mean(), 4),
                 "cakupan terburuk": round(coverage_by_group[name].min(), 4)})
table = pd.DataFrame(rows, index=list(METHODS))
table.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv")
BEST = "ternormalisasi"        # GANTI: pilih dari tabel (cakupan ≈ target, lebar kecil)
print(table, "\ntarget cakupan:", 1 - ALPHA)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(coverage_by_group).plot.bar(ax=axes[0], color=["#0072B2", "#E69F00", "#009E73"])
axes[0].axhline(1 - ALPHA, color="#1a1a1a", linewidth=1, linestyle="--")
axes[0].set_xlabel("kelompok")
axes[0].set_ylabel("cakupan")
axes[0].tick_params(axis="x", rotation=0)
picked = np.sort(rng.choice(n_train, 120, replace=False))
picked = picked[np.argsort(oof[picked])]
low, high = METHODS[BEST](np.arange(n_train), picked)
axes[1].fill_between(np.arange(len(picked)), low, high, color="#0072B2", alpha=0.25)
axes[1].plot(np.arange(len(picked)), oof[picked], color="#0072B2", linewidth=1)
axes[1].scatter(np.arange(len(picked)), y[picked], s=8, color="#D55E00")
axes[1].set_xlabel("baris (urut tebakan)")
axes[1].set_ylabel("target (skala model)")
axes[1].set_title(f"interval {int((1 - ALPHA) * 100)}% ({BEST})")
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/cakupan_dan_contoh.png", dpi=200, bbox_inches="tight")
plt.show()

## Interval test + baris paling tidak yakin (bahan slide)

In [ ]:
def to_target_scale(values):
    if LOG_TARGET:
        values = np.expm1(values)
    if CLIP_TO_RANGE:
        values = np.clip(values, target.min(), target.max())
    return values


low, high = METHODS[BEST](np.arange(n_train), np.arange(n_train, len(pred_all)))
result = pd.DataFrame()
result[ID_COL] = test[ID_COL]
result["tebakan"] = to_target_scale(test_pred)
result["bawah"] = to_target_scale(low)
result["atas"] = to_target_scale(high)
result["lebar"] = result["atas"] - result["bawah"]
result.to_csv(f"{OUTPUT_FOLDER}/interval_test.csv", index=False)
unsure = result.sort_values("lebar", ascending=False).head(20)
unsure = unsure.assign(teks=full_test.to_numpy()[unsure.index.to_numpy()])
unsure["teks"] = unsure["teks"].str.slice(0, 150)
unsure.to_csv(f"{OUTPUT_FOLDER}/paling_tidak_yakin.csv", index=False)
print(result.describe().round(3))

## Cek cakupan test dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = test[ID_COL].map(key_by_id).to_numpy()
for name in METHODS:
    low, high = METHODS[name](np.arange(n_train), np.arange(n_train, len(pred_all)))
    low = to_target_scale(low)
    high = to_target_scale(high)
    inside = (truth >= low) & (truth <= high)
    print(f"{name:15s} | cakupan test {inside.mean():.4f} | lebar {np.mean(high - low):.4f}")